# Mamba-3 MiniGrid Memory — Kaggle T4 双实验

Parallel training on **NVIDIA T4 x2**:
- **实验 A**: `MiniGrid-MemoryS11-v0` — quick curriculum validation
- **实验 B**: `MiniGrid-MemoryS17Random-v0` — long-range memory challenge

Setup: GPU T4x2, Internet ON, ~12h session.

In [ ]:
# ===== Configuration =====
from pathlib import Path
import os

REPO_URL  = "https://github.com/SShion0721/mamba-minigrid-memory.git"
REPO_BRANCH = "main"
WORKDIR = Path("/kaggle/working/mamba-minigrid-memory")
RUNS_DIR = WORKDIR / "runs"

EXP_A = dict(name="mamba3_s11_seed42",        env_id="MiniGrid-MemoryS11-v0",        total_steps=5_000_000,  seed=42)
EXP_B = dict(name="mamba3_s17random_seed42",  env_id="MiniGrid-MemoryS17Random-v0",  total_steps=10_000_000, seed=42)

# Shared
VARIANT    = "mamba3"
SPATIAL    = "hybrid"
NUM_ENVS   = 64       # T4 16GB safe value; try 96-128 if stable
NUM_STEPS  = 256
CONTEXT_LEN = 128
CHUNK_LEN  = 128
BATCH_CHUNKS = 32
D_MODEL    = 256
MAMBA_LAYERS = 4
D_STATE    = 16

LR         = 1e-4
ENT_COEF   = 0.01
ENT_COEF_FINAL = 0.001
EVAL_INTERVAL = 20_000
EVAL_EPISODES = 30
SAVE_INTERVAL = 100_000
LOG_INTERVAL  = 5

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")

In [ ]:
# ===== Environment check =====
import subprocess, sys

def run(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env,
                          stdout=sys.stdout, stderr=sys.stderr)

run("nvidia-smi", check=False)
print(f"Python: {sys.version}")

In [ ]:
# ===== Install dependencies =====
run(f"{sys.executable} -m pip install -q --upgrade pip")
run(f"{sys.executable} -m pip install -q gymnasium minigrid tensorboard tqdm einops matplotlib imageio[ffmpeg]")
print("Installing mamba-ssm ...")
run(f"{sys.executable} -m pip install -q --no-cache-dir causal-conv1d mamba-ssm --no-build-isolation")

import torch, numpy as np
num_gpus = torch.cuda.device_count()
print(f"torch={torch.__version__}  cuda={torch.version.cuda}  GPU count={num_gpus}")
for i in range(num_gpus):
    prop = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {prop.name}  {prop.total_mem/1e9:.1f} GB")

# Validate Mamba-3
try:
    from mamba_ssm import Mamba3
except ImportError:
    from mamba_ssm.modules.mamba3 import Mamba3
print(f"Mamba3 OK  Mamba3={Mamba3 is not None}")

In [ ]:
# ===== Clone repo =====
import shutil
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {WORKDIR}")
assert (WORKDIR / "src" / "train_mamba_ppo.py").exists()
print("Code ready.")

In [ ]:
# ===== Launch parallel training with live output =====
import subprocess, sys, time, os, threading, io
from pathlib import Path

os.chdir(WORKDIR)
logs = {}  # name -> io.StringIO with full captured output

def build_cmd(exp):
    exe = sys.executable
    args = [
        exe, "-u", str(WORKDIR / "src" / "train_mamba_ppo.py"),
        "--model", "mamba", "--mamba-variant", VARIANT,
        "--env-id", exp["env_id"], "--seed", str(exp["seed"]),
        "--total-steps", str(exp["total_steps"]),
        "--num-envs", str(NUM_ENVS), "--num-steps", str(NUM_STEPS),
        "--context-len", str(CONTEXT_LEN), "--chunk-len", str(CHUNK_LEN),
        "--batch-chunks", str(BATCH_CHUNKS),
        "--d-model", str(D_MODEL), "--mamba-layers", str(MAMBA_LAYERS),
        "--d-state", str(D_STATE),
        "--spatial-encoder", SPATIAL,
        "--spatial-layers", "2", "--spatial-heads", "4",
        "--valid-actions", "0,1,2",
        "--lr", str(LR), "--ent-coef", str(ENT_COEF),
        "--ent-coef-final", str(ENT_COEF_FINAL),
        "--eval-interval", str(EVAL_INTERVAL),
        "--eval-episodes", str(EVAL_EPISODES),
        "--save-interval", str(SAVE_INTERVAL),
        "--log-interval", str(LOG_INTERVAL),
        "--run-name", exp["name"],
    ]
    return args


def run_one(label, exp, gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = build_cmd(exp)

    print(f"\n{'='*60}")
    print(f"[{label}] {exp['name']}  env={exp['env_id']}  steps={exp['total_steps']:,}")
    print(f"[{label}] GPU={gpu_id}  envs={NUM_ENVS}  ctx={CONTEXT_LEN}")
    print(f"[{label}] d_model={D_MODEL}  layers={MAMBA_LAYERS}")
    print(f"{'='*60}")

    buf = io.StringIO()
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    start = time.time()
    line_count = 0
    for line in proc.stdout:
        line = line.rstrip()
        buf.write(line + "\n")
        line_count += 1
        # Print all training/eval lines (not the tqdm progress spam)
        if any(kw in line for kw in [
            "Eval step", "Saved checkpoint", "Training complete",
            "Update", "train_return", "train_success", "SPS",
            "Training config", "Using device", "Resumed",
            "Error", "Traceback", "exception", "OOM"
        ]):
            print(f"[{label}] {line}")
    proc.wait()
    elapsed = time.time() - start
    ok = proc.returncode == 0
    print(f"\n[{label}] {'PASS' if ok else 'FAIL'}  exit={proc.returncode}  time={elapsed/3600:.1f}h")
    logs[label] = buf.getvalue()
    return proc.returncode


if num_gpus >= 2:
    print("Dual GPU mode")
    import threading
    results = {}
    def ra(): results["A"] = run_one("EXP-A", EXP_A, 0)
    def rb(): results["B"] = run_one("EXP-B", EXP_B, 1)
    ta = threading.Thread(target=ra); tb = threading.Thread(target=rb)
    ta.start(); time.sleep(3); tb.start()
    ta.join(); tb.join()
else:
    print("Single GPU mode (serial)")
    run_one("EXP-A", EXP_A, 0)
    run_one("EXP-B", EXP_B, 0)

# Print full error log if any experiment failed
for label, name in [("EXP-A", EXP_A["name"]), ("EXP-B", EXP_B["name"])]:
    if label in logs:
        full_log = logs[label]
        if "Traceback" in full_log or "Error" in full_log:
            lines = full_log.strip().split("\n")
            print(f"\n{'='*60}")
            print(f"[{label}] FULL ERROR LOG ({len(lines)} lines)")
            print(f"{'='*60}")
            for line in lines[-50:]:  # last 50 lines
                print(line)

In [ ]:
# ===== TensorBoard curves inline =====
# Reads the tfevents files directly and plots success_rate and return

%matplotlib inline
import matplotlib.pyplot as plt
from collections import defaultdict
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = {"EXP-A": "tab:blue", "EXP-B": "tab:orange"}

for label, exp in [("EXP-A", EXP_A), ("EXP-B", EXP_B)]:
    run_dir = RUNS_DIR / exp["name"]
    if not run_dir.exists():
        print(f"{label}: run dir not found")
        continue
    event_files = sorted(run_dir.glob("events.out.tfevents.*"))
    if not event_files:
        print(f"{label}: no tfevents files")
        continue
    
    ea = EventAccumulator(str(event_files[-1]))
    ea.Reload()
    
    for tag in ["eval/success_rate", "eval/mean_return"]:
        if tag in ea.Tags()["scalars"]:
            steps = []
            values = []
            for evt in ea.Scalars(tag):
                steps.append(evt.step)
                values.append(evt.value)
            if tag == "eval/success_rate":
                ax1.plot(steps, values, color=colors[label], label=f"{label}", marker=".")
            else:
                ax2.plot(steps, values, color=colors[label], label=f"{label}", marker=".")
    print(f"{label}: event file loaded, {len(event_files)} events")

ax1.set_title("eval/success_rate"); ax1.set_xlabel("global_step"); ax1.legend(); ax1.grid(True)
ax2.set_title("eval/mean_return");  ax2.set_xlabel("global_step"); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Final evaluation (100 episodes, greedy) =====
import sys; sys.path.insert(0, str(WORKDIR))
from src.eval import evaluate as eval_checkpoint

for label, exp in [("EXP-A", EXP_A), ("EXP-B", EXP_B)]:
    run_dir = RUNS_DIR / exp["name"]
    ckpt = run_dir / "model_latest.pt"
    if not ckpt.exists():
        ckpts = sorted(run_dir.glob("model_*.pt"))
        ckpt = ckpts[-1] if ckpts else None
    if ckpt is None:
        print(f"{label}: NO CHECKPOINT — training failed")
        continue
    print(f"\n{'='*50}")
    print(f"{label}: {ckpt.name}")
    print(f"{'='*50}")
    eval_checkpoint(str(ckpt), episodes=100, seed=999, deterministic=True)

In [ ]:
# ===== Visualize agent behavior (3 episodes each) =====
from src.visualize import record_episodes

for label, exp in [("EXP-A", EXP_A), ("EXP-B", EXP_B)]:
    run_dir = RUNS_DIR / exp["name"]
    ckpt = run_dir / "model_latest.pt"
    if not ckpt.exists():
        ckpts = sorted(run_dir.glob("model_*.pt"))
        ckpt = ckpts[-1] if ckpts else None
    if ckpt is None:
        continue
    print(f"\n{'='*50}")
    print(f"{label}: {exp['env_id']}")
    print(f"{'='*50}")
    record_episodes(str(ckpt), num_episodes=3, seed=42,
                    save_dir=str(WORKDIR / "videos" / exp["name"]),
                    fps=8, deterministic=True)